# Practice Lab: BERT and Encoder Models (Lesson 4.3)

Module 4 · 6 exercises · ~90-120 min
**Needs:** transformers, datasets, scikit-learn


# Lesson 4.3: BERT and Encoder Models

6 exercises: 2 Easy + 2 Medium + 2 Hard
**Needs:** `pip install transformers datasets scikit-learn`


In [ ]:
!pip install transformers datasets scikit-learn -q


---
## Exercise 1 (Easy): Explore MLM + Bias


In [ ]:
# YOUR CODE
from transformers import pipeline

unmasker = pipeline("fill-mask", model="bert-base-uncased")

sentences = [
    "The capital of India is [MASK].",
    "Python is a popular [MASK] language.",
    "I ordered butter [MASK] for dinner.",
    "The [MASK] chased the mouse.",
    "This man works as a [MASK].",
    "This woman works as a [MASK].",
]

for sent in sentences:
    results = unmasker(sent)[:3]
    print(f"\n  {sent}")
    for r in results:
        print(f"    -> '{r['token_str']}' ({r['score']:.0%})")

# TODO: analyze man vs woman bias


<details><summary>Solution</summary>


In [ ]:
from transformers import pipeline

unmasker = pipeline("fill-mask", model="bert-base-uncased")

sentences = [
    "The capital of India is [MASK].",
    "Python is a popular [MASK] language.",
    "I ordered butter [MASK] for dinner.",
    "After winning, they felt [MASK].",
    "The [MASK] chased the mouse.",
    "This man works as a [MASK].",
    "This woman works as a [MASK].",
    "The [MASK] nurse helped the patient.",
    "The [MASK] engineer designed the bridge.",
    "People from [MASK] are very friendly.",
]

for sent in sentences:
    results = unmasker(sent)[:3]
    print(f"\n  {sent}")
    for r in results:
        print(f"    -> '{r['token_str']}' ({r['score']:.0%})")

# Bias analysis
print("\n=== BIAS ANALYSIS ===")
for gender in ["man", "woman"]:
    sent = f"This {gender} works as a [MASK]."
    results = unmasker(sent)[:5]
    print(f"\n  '{gender}' top jobs:")
    for r in results:
        print(f"    {r['token_str']:15s} {r['score']:.0%}")

print("\nBiases found:")
print("  - Gender stereotypes in job predictions")
print("  - 'man' -> lawyer/doctor, 'woman' -> nurse/maid")
print("  - Production: audit, debias, add guardrails")


</details>


---
## Exercise 2 (Easy): Compare Models


In [ ]:
# YOUR CODE
from transformers import pipeline
import time

models = [
    ("BERT", "textattack/bert-base-uncased-SST-2"),
    ("DistilBERT",
     "distilbert-base-uncased-finetuned-sst-2-english"),
    ("RoBERTa", "textattack/roberta-base-SST-2"),
]

reviews = [
    "Absolutely brilliant! The acting was superb.",
    "Worst movie ever. Complete waste of time.",
    "It was okay. Nothing special.",
    "Visuals stunning but story was weak.",
    "A masterpiece. Will watch again!",
]

# TODO: time each model, print comparison table


<details><summary>Solution</summary>


In [ ]:
from transformers import pipeline
import time

models = [
    ("BERT", "textattack/bert-base-uncased-SST-2"),
    ("DistilBERT",
     "distilbert-base-uncased-finetuned-sst-2-english"),
    ("RoBERTa", "textattack/roberta-base-SST-2"),
]

reviews = [
    "Absolutely brilliant! The acting was superb.",
    "Worst movie ever. Complete waste of time.",
    "It was okay. Nothing special.",
    "Visuals stunning but story was weak.",
    "A masterpiece. Will watch again!",
]

results = {}
for name, model_id in models:
    pipe = pipeline("sentiment-analysis", model=model_id)
    t0 = time.time()
    preds = pipe(reviews)
    ms = (time.time() - t0) / len(reviews) * 1000
    results[name] = {"ms": ms, "preds": preds}
    print(f"  {name:12s}: {ms:.0f}ms/review")

print("\nPer-review comparison:")
for i, rev in enumerate(reviews):
    print(f"\n  '{rev[:40]}...'")
    for name in results:
        p = results[name]["preds"][i]
        print(f"    {name:12s}: "
              f"{p['label']:>8s} ({p['score']:.0%})")

print("\nBest production choice: DistilBERT")
print("  40% smaller, 60% faster, 95% accuracy")


</details>


---
## Exercise 4 (Medium): [CLS] Vector Explorer


In [ ]:
# YOUR CODE
from transformers import BertTokenizer, BertModel
import torch
import numpy as np

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")
model.eval()

sentences = [
    "The team won the championship game.",
    "He scored a hat trick in football.",
    "Cricket is India's favorite sport.",
    "Biryani is a delicious rice dish.",
    "I ordered butter chicken for dinner.",
    "Fresh dosa with coconut chutney.",
    "Python is a popular programming language.",
    "Machine learning predicts outcomes.",
    "GPUs accelerate neural network training.",
    "The acting in this film was superb.",
    "I cried during the emotional scenes.",
    "Best movie I have seen this year.",
]

def get_cls(text):
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, max_length=128)
    with torch.no_grad():
        out = model(**inputs)
    return out.last_hidden_state[0, 0].numpy()

# TODO: extract [CLS] vectors, compute similarity
# TODO: plot heatmap


<details><summary>Solution</summary>


In [ ]:
from transformers import BertTokenizer, BertModel
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")
model.eval()

sentences = [
    "The team won the championship game.",
    "He scored a hat trick in football.",
    "Cricket is India's favorite sport.",
    "Biryani is a delicious rice dish.",
    "I ordered butter chicken for dinner.",
    "Fresh dosa with coconut chutney.",
    "Python is a popular programming language.",
    "Machine learning predicts outcomes.",
    "GPUs accelerate neural network training.",
    "The acting in this film was superb.",
    "I cried during the emotional scenes.",
    "Best movie I have seen this year.",
]

topics = ["Sport"]*3 + ["Food"]*3 + ["Tech"]*3 + ["Movie"]*3

def get_cls(text):
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, max_length=128)
    with torch.no_grad():
        out = model(**inputs)
    return out.last_hidden_state[0, 0].numpy()

vecs = np.array([get_cls(s) for s in sentences])
sim = cosine_similarity(vecs)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim, cmap='RdYlGn', vmin=0.85, vmax=1.0)
for i, t in enumerate(topics):
    ax.text(-0.5, i, t, ha='right', fontsize=7, va='center')
plt.colorbar(im)
ax.set_title('[CLS] Cosine Similarity')
plt.tight_layout()
plt.savefig('cls_similarity.png', dpi=150)
plt.show()

# Within vs between topic
topic_ids = [0]*3 + [1]*3 + [2]*3 + [3]*3
within, between = [], []
for i in range(12):
    for j in range(i+1, 12):
        if topic_ids[i] == topic_ids[j]:
            within.append(sim[i,j])
        else:
            between.append(sim[i,j])
print(f"Avg within-topic:  {np.mean(within):.3f}")
print(f"Avg between-topic: {np.mean(between):.3f}")


</details>


---
## Exercise 3 (Medium): Hindi Sentiment with IndicBERT
See practice-lab-4_3.html for full details.


In [ ]:
# Exercise 3: Hindi Sentiment with IndicBERT
# See practice-lab-4_3.html for complete
# starter code, hints, expected output.
# Needs Colab with HuggingFace transformers.
pass


---
## Exercise 5 (Hard): Custom News Classifier (ag_news)
See practice-lab-4_3.html for full details.


In [ ]:
# Exercise 5: Custom News Classifier (ag_news)
# See practice-lab-4_3.html for complete
# starter code, hints, expected output.
# Needs Colab with HuggingFace transformers.
pass


---
## Exercise 6 (Challenge): Named Entity Recognition
See practice-lab-4_3.html for full details.


In [ ]:
# Exercise 6: Named Entity Recognition
# See practice-lab-4_3.html for complete
# starter code, hints, expected output.
# Needs Colab with HuggingFace transformers.
pass
